# Complete MLA integration

This notebook places naive MLA inside a dense Transformer block and compares its output shape and parameter count with the GPT baseline.

In [ ]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(7)

## Compare the complete models

Both models keep the same embeddings, FFN, residual structure, final norm, and language-model head. Only the attention implementation changes.

In [ ]:
from src.model import DeepSeekConfig, DeepSeekMLA, DeepSeekSkeleton

config = DeepSeekConfig(
    vocab_size=256,
    dim=64,
    n_layers=2,
    n_heads=4,
    q_lora_rank=32,
    kv_lora_rank=16,
    qk_nope_head_dim=16,
    qk_rope_head_dim=8,
    v_head_dim=16,
    inter_dim=128,
    max_seq_len=32,
)
gpt = DeepSeekSkeleton(config)
mla = DeepSeekMLA(config)
tokens = torch.randint(0, config.vocab_size, (2, 8))

gpt_logits = gpt(tokens)
mla_logits = mla(tokens)
print("GPT logits:", gpt_logits.shape)
print("MLA logits:", mla_logits.shape)

In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())

print("GPT parameters:", count_parameters(gpt))
print("MLA parameters:", count_parameters(mla))

loss = mla_logits.square().mean()
loss.backward()
assert mla.blocks[0].attention.q_branch.q_down.weight.grad is not None
assert mla.blocks[-1].attention.kv_branch.kv_down.weight.grad is not None
print("full-model backward check passed")

The naive MLA model is now a complete trainable dense Transformer. It still materializes expanded keys and values; compressed-cache decoding comes later.